<a href="https://colab.research.google.com/github/namita-code/6thSEM-ML-LAB/blob/main/1BM23CS202_Lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

# ---------------------------
# 1. Load dataset
# ---------------------------
df = pd.read_csv("heart.csv")

# Assume target column is named 'target' (adjust if needed)
target_col = 'HeartDisease'

X = df.drop(columns=[target_col])
y = df[target_col]

# ---------------------------
# 2. Identify categorical & numerical columns
# ---------------------------
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# ---------------------------
# 3. Preprocessing
# - OneHotEncoding for categorical
# - Scaling for numerical
# ---------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# ---------------------------
# 4. Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# 5. Define models
# ---------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier()
}

# ---------------------------
# 6. Train & evaluate models (without PCA)
# ---------------------------
print("=== Without PCA ===")
results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"{name}: {acc:.4f}")

best_model_name = max(results, key=results.get)
print(f"\nBest model (without PCA): {best_model_name}")

# ---------------------------
# 7. Apply PCA
# ---------------------------
# Choose number of components (or variance ratio)
pca = PCA(n_components=0.95)  # keeps 95% variance

print("\n=== With PCA ===")
pca_results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('pca', pca),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    pca_results[name] = acc

    print(f"{name} (with PCA): {acc:.4f}")

best_pca_model = max(pca_results, key=pca_results.get)
print(f"\nBest model (with PCA): {best_pca_model}")

# ---------------------------
# 8. Comparison summary
# ---------------------------
print("\n=== Comparison ===")
for model in models.keys():
    print(f"{model}: Without PCA = {results[model]:.4f}, With PCA = {pca_results[model]:.4f}")

=== Without PCA ===
Logistic Regression: 0.8533
SVM: 0.8641
Random Forest: 0.8804

Best model (without PCA): Random Forest

=== With PCA ===
Logistic Regression (with PCA): 0.8533
SVM (with PCA): 0.8587
Random Forest (with PCA): 0.8696

Best model (with PCA): Random Forest

=== Comparison ===
Logistic Regression: Without PCA = 0.8533, With PCA = 0.8533
SVM: Without PCA = 0.8641, With PCA = 0.8587
Random Forest: Without PCA = 0.8804, With PCA = 0.8696
